In [ ]:
#2. 고객 페르소나 기반 맞춤 이메일 시퀀스
#목표: Toss 고객 페르소나(Persona)를 설정하고, 상황별 맞춤 이메일 자동화
#실습 내용:
#데이터 기반으로 Toss 고객 페르소나 2개 작성 (남 1 / 여 1)
#페르소나별 주요 Pain Point와 Needs 정리
#ConvertKit에서 페르소나별 이메일 시퀀스 제작
#Zapier를 활용해 페르소나별 행동(예: 설문 응답 결과)에 따라 이메일 자동 발송
#성과물: 페르소나 시트 (*엑셀) + ConvertKit 시퀀스 설계 (*캡처)

In [9]:
#카테고리별 리뷰 요약
#W Concept 내 카테고리별 리뷰 데이터 크롤링을 통해 "카테고리별 평균 평점과 리뷰 수"를 구하고, 오늘배운 CASE WHEN절을 활용해서 Excellent / Average / Poor로 분류하세요.
#카테고리 선정은 본인 자유 입니다.
#크롤링 상품 개수는 최소 30개입니다.
#Excellent / Average / Poor의 기준은 본인 자유 입니다.

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re 
import time
import pandas as pd
import pymysql

service = Service(ChromeDriverManager().install())
options = Options()
#options.add_argument('--headless')
options.add_argument('--window-size=1920x1080')
options.add_argument('--lang=ko_KR')
options.add_argument('--start-maximized')
options.add_argument('--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36')

driver = webdriver.Chrome(service=service, options=options)
url = "https://display.wconcept.co.kr/rn/women"
driver.get(url)

try:
    best_button = driver.find_element(By.XPATH, "//button[text()='베스트']")
    best_button.click()
    time.sleep(2)
except Exception as e:
    print(f"베스트 버튼 클릭 실패 :{e}")

try:
    outer_button = driver.find_element(By.XPATH, "//button[text()='아우터']")
    outer_button.click()
    time.sleep(2)
except Exception as e:
    print(f"아우터 버튼 클릭실패! : {e}")

scroll_pause = 2
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(scroll_pause)
time.sleep(3)

main_window_handle = driver.current_window_handle

w_outer = driver.find_elements(By.CSS_SELECTOR, "div.product-item")[:30]
print(f"{len(w_outer)}개 상품 정보를 수집하겠습니당")

w_product_data = []

for item in w_outer:
    try:
        brand = item.find_element(By.CSS_SELECTOR, "span.prdc-title > span.text.title").text.strip()
        outer_name = item.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()
        outer_price = item.find_element(By.CSS_SELECTOR, "span.text.final-price").text.strip()
        num_outer_price = int(outer_price.replace(',', ''))

        try:
            img_el = item.find_element(By.CSS_SELECTOR, "img")
            img_src = img_el.get_attribute("src") or ""
            m = re.search(r"/(\d{8,12})[^\d]", img_src)
            goods_no = m.group(1) if m else None
            detail_url = f"https://www.wconcept.co.kr/Product/{goods_no}" if goods_no else None
        except Exception:
            detail_url = None

        discount_percent = 0
        try:
            outer_discount = item.find_element(By.CSS_SELECTOR, "span.text.final-discount > em").text.strip()
            # 정규식으로 숫자만 추출
            m = re.search(r"\d+", outer_discount)
            discount_text = m.group() if m else "0"
            discount_percent = int(discount_text)
        except NoSuchElementException:
            pass

        w_product_data.append({
            "brand": brand,
            "outer_name": outer_name,
            "outer_price": num_outer_price,
            "outer_discount": discount_percent,
            "detail_url": detail_url
        })

    except Exception as e:
        print(f"상품 정보 수집 중 에러 발생: {e}")

for newtab in w_product_data:
    review_count = 0
    try:
        driver.execute_script("window.open(arguments[0]);", newtab['detail_url'])
        new_window_handle = [h for h in driver.window_handles if h != main_window_handle][0]
        driver.switch_to.window(new_window_handle)
        time.sleep(2)

        try:
            stargrade_txt = driver.find_element(By.CSS_SELECTOR, "div.review_info p.grade_num").text.strip()
            review_txt = driver.find_element(By.CSS_SELECTOR, "div.review_info em#reviewCnt1").text.strip()

            stargrade = re.sub(r"[^\d\.]", "", stargrade_txt)
            review_count = re.sub(r"[^\d]", "", review_txt) if review_txt else 0

            newtab['stargrade'] = float(stargrade) if stargrade else 0
            newtab['review_count'] = int(review_count) if review_count else 0

        except Exception:
            newtab['stargrade'] = 0
            newtab['review_count'] = 0

        if newtab['review_count'] == 0 or newtab['stargrade'] == 0:
            print("리뷰 없음: 스킵", newtab.get('detail_url'))
            try:
                driver.close()
                driver.switch_to.window(main_window_handle)
                time.sleep(1)
            except:
                pass
            continue

        # >>> 추가: 리뷰 있으면 리뷰 영역 클릭
        try:
            review_info = driver.find_element(By.CSS_SELECTOR, "div.review_info")
            review_info.click()
            time.sleep(2)

            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            review_elements = driver.find_elements(By.CSS_SELECTOR, "p.pdt_review_text")
            reviews = []
            for review in review_elements:
                text = review.get_attribute("innerText").strip()
                if text:
                    reviews.append(text)
            newtab["review_text"] = " || ".join(reviews[:10])
            newtab["reviews"] = reviews

        except Exception as e:
            print("리뷰 클릭 실패:", e)
            try:
                driver.close()
                driver.switch_to.window(main_window_handle)
                time.sleep(1)
            except:
                pass
            continue

        finally:
            try:
                driver.close()
                driver.switch_to.window(main_window_handle)
                time.sleep(1)
            except:
                pass

    except Exception:   
        newtab['stargrade'] = 0
        newtab['review_count'] = 0

df = pd.DataFrame(w_product_data)
display(df.head(10))
print(f"총 상품수: {len(df)}")
print(f"평균 평점: {df['stargrade'].dropna().mean():.2f}" if df['stargrade'].notna().any() else "평균 평점: N/A")
print(f"총 리뷰수: {df['review_count'].sum()}")

# ===== 여기서부터 MySQL 저장 =====
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='Wonsil929621!@#',
    db='wconcep',   
    port=3306,
    charset='utf8mb4'
)

try:
    with conn.cursor() as cur:
        sql = """
        INSERT INTO w_outer_info4
        (brand, outer_name, outer_price, outer_discount, stargrade, review_count)
        VALUES (%s, %s, %s, %s, %s, %s)
        """
        rows = [
            (
                item.get("brand"),
                item.get("outer_name"),
                item.get("outer_price"),
                item.get("outer_discount"),
                item.get("stargrade"),
                item.get("review_count")
            )
            for item in w_product_data
        ]
        cur.executemany(sql, rows)
        conn.commit()
        print(f"{len(rows)} rows inserted into MySQL successfully!")

        sql_review = """ 
        INSERT INTO w_outer_review (product_id, review_text)
        VALUES (%s, %s)
        """
        for item in w_product_data:
            reviews = item.get("reviews", [])
            if not reviews:
                continue

            # 상품 id 찾기 (brand + outer_name 기준, 가장 최근 것)
            cur.execute(
                "SELECT id FROM w_outer_info4 WHERE brand=%s AND outer_name=%s ORDER BY id DESC LIMIT 1",
                (item.get("brand"), item.get("outer_name"))
            )
            result = cur.fetchone()
            if not result:
                continue
            product_id = result[0]

            # 리뷰 여러 개 insert
            for review in reviews:
                cur.execute(sql_review, (product_id, review))

        conn.commit()
        print("리뷰들도 DB에 저장 완료!")

except Exception as e:
    print("DB insert error:", e)
finally:
    conn.close()


30개 상품 정보를 수집하겠습니당
리뷰 없음: 스킵 https://www.wconcept.co.kr/Product/307377173
리뷰 없음: 스킵 https://www.wconcept.co.kr/Product/307329760
리뷰 없음: 스킵 https://www.wconcept.co.kr/Product/306104850
리뷰 없음: 스킵 https://www.wconcept.co.kr/Product/307378028
리뷰 없음: 스킵 https://www.wconcept.co.kr/Product/307367879
리뷰 없음: 스킵 https://www.wconcept.co.kr/Product/307349797
리뷰 없음: 스킵 https://www.wconcept.co.kr/Product/307378529


,brand,outer_name,outer_price,outer_discount,detail_url,stargrade,review_count,review_text,reviews
0,던스트,[15%+5%쿠폰] [단독]UNISEX WAXED COTTON HUNTING JAC...,269100,10,https://www.wconcept.co.kr/Product/307335695,5.0,8,브라운 색상으로 사이즈 진짜 고민 많이 했는데키가 커서 m했는데 아주 찰떡 ! 평소...,[브라운 색상으로 사이즈 진짜 고민 많이 했는데키가 커서 m했는데 아주 찰떡 ! 평...
1,그노노이,[30%쿠폰] 버튼 레더 하프자켓 (Black),179690,30,https://www.wconcept.co.kr/Product/307377173,0.0,0,NaN,NaN
2,룩캐스트,[72H특가] [단독]에일린 울 핸드메이드 자켓 / AILEEN WOOL HANDM...,132648,33,https://www.wconcept.co.kr/Product/307329760,0.0,0,NaN,NaN
3,더라우스트,[정린 PICK!] 유틸리티 드로우스트링 트렌치 점퍼 KHAKI,81210,42,https://www.wconcept.co.kr/Product/307360523,5.0,4,좋다...가볍다.....가을가을하다... || 소재가 맘에들어요소매를좀 접어야 핏이...,"[좋다...가볍다.....가을가을하다..., 소재가 맘에들어요소매를좀 접어야 핏이사..."
4,제이청,Wet Leather Blouson [3COL],224342,29,https://www.wconcept.co.kr/Product/307319127,5.0,1,제이청 옷을 입어보고 좋아서 자켓류로 하나 더 사봅니다 역시 옷을 잘 만드는 브랜드...,[제이청 옷을 입어보고 좋아서 자켓류로 하나 더 사봅니다 역시 옷을 잘 만드는 브랜...
5,프론트로우,[Day-Wool] Handmade Duffle Coat_3color,133138,66,https://www.wconcept.co.kr/Product/301406575,4.9,631,색은 보기와 같아요. 큼직한데 겨울이라 안에입는 소재도두꺼우니까 많이 클거같지는 않...,[색은 보기와 같아요. 큼직한데 겨울이라 안에입는 소재도두꺼우니까 많이 클거같지는 ...
6,온앤온,Boucle High-neck Half Coat NEW4WHA23_03,239950,52,https://www.wconcept.co.kr/Product/306176282,4.8,18,세일해서 미리 주문했어요귀엽고 저렴히 잘 산듯 해요 || 가격이나 리뷰때문에 기대를...,"[세일해서 미리 주문했어요귀엽고 저렴히 잘 산듯 해요, 가격이나 리뷰때문에 기대를 ..."
7,망고매니플리즈,[PRE-OPEN] NU LEATHER JACKET,234550,18,https://www.wconcept.co.kr/Product/306061260,4.9,163,원피스에 입어보려고 샀어요.핏이 생각보다 더 이쁘게 빠지더라구요아주 잘 산거 같아요...,[원피스에 입어보려고 샀어요.핏이 생각보다 더 이쁘게 빠지더라구요아주 잘 산거 같아...
8,나이스클랍,나이스클랍 벨티드 롱다운 X246PWDA04,171140,76,https://www.wconcept.co.kr/Product/306104850,0.0,0,NaN,NaN
9,랭앤루,[한정특가] [단독] NOTTINGHILL Washing Faux Leather B...,127883,41,https://www.wconcept.co.kr/Product/304164628,4.9,207,다크 브라운이라서 사실 거의 검정에 가깝다 싶긴 한데요막입기 편하고 아주 좋았습니다...,[다크 브라운이라서 사실 거의 검정에 가깝다 싶긴 한데요막입기 편하고 아주 좋았습니...


총 상품수: 30
평균 평점: 3.79
총 리뷰수: 1789
30 rows inserted into MySQL successfully!
리뷰들도 DB에 저장 완료!
